# Severity CNN (ResNet-18) — PothRGBD tiers (Component 6.3B)

Trains the severity head (Contribution 2) on RGB crops of clean PothRGBD potholes to predict
low/med/high, and compares against the box-area heuristic baseline on the SAME splits.

> **Fair-claim note.** The CNN input is the **RGB crop ONLY** — depth was used only to derive
> the tier labels, never as a model input. Every crop is **resized to 224x224**, so absolute
> crop size / box area is **not** a usable feature: the model must use **texture/shading**, not
> size. That is what makes "the CNN beats box-area" an honest claim (box-area is the baseline).


## Cell 1 - clone/pull, install, GPU check

In [ ]:
import os
REPO_URL = "https://github.com/YashChugani/adaptive-tile-pothole-detection.git"  # <-- edit if needed (PAT for private)
REPO_DIR = "/content/Pothole-Detection-System"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
# torch/torchvision/sklearn are preinstalled on Colab; keep Colab's GPU torch.
!pip install -q ultralytics==8.4.160 sahi==0.12.6
%cd {REPO_DIR}
import torch
assert torch.cuda.is_available(), "No CUDA GPU. Runtime > Change runtime type > GPU."
print("torch", torch.__version__, "| CUDA GPU:", torch.cuda.get_device_name(0))


## Cell 2 - severity setup (Drive mount, crops to local SSD, checkpoint dir)

In [ ]:
import sys
sys.path.insert(0, "/content/Pothole-Detection-System/notebooks")
import severity_setup
cfg = severity_setup.setup()
cfg


## Cell 3 - train ResNet-18 (AUTHOR ONLY; this is the long run)

ImageNet-pretrained. Resize->224 (size is not a feature). ImageNet normalization. Light,
depth-preserving augmentation (small rotation, hflip, mild color/brightness jitter; NO
aggressive crops/zooms). **Overfitting guards** for 618 train crops: `weight_decay=1e-4`,
`Dropout(0.4)` before the FC head, and TRAIN acc/macro-F1 printed next to VAL every epoch so
the train/val gap is visible. Runs **both** `TRAIN_MODES` in sequence (full fine-tune vs frozen
backbone + FC head), each early-stopping on its own VAL macro-F1; cell 4 uses whichever wins.

In [ ]:
import torch, torch.nn as nn, numpy as np
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import f1_score, accuracy_score

# ---------------- PARAMS ----------------
CROPS   = Path(cfg["crops_root"])
PROJECT = Path(cfg["project"]); PROJECT.mkdir(parents=True, exist_ok=True)
TRAIN_MODES = ("finetune_all", "frozen_backbone")   # both are run; best-on-val is evaluated
EPOCHS  = 40
PATIENCE= 8          # early stop on val macro-F1
LR      = 1e-4       # low LR
WD      = 1e-4       # weight decay (overfitting guard)
DROPOUT = 0.4        # before the FC head (overfitting guard)
BATCH   = 32
IMG     = 224
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]
# ----------------------------------------

train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),                      # force size out as a feature
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_ds = datasets.ImageFolder(CROPS/"pothrgbd"/"train", train_tf)
val_ds   = datasets.ImageFolder(CROPS/"pothrgbd"/"val",   eval_tf)
CLASSES  = train_ds.classes                              # alphabetical: ['high','low','medium']
print("classes:", CLASSES, "| train", len(train_ds), "val", len(val_ds))
train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=2)
val_dl   = DataLoader(val_ds, BATCH, shuffle=False, num_workers=2)
device = "cuda"

def build_model(mode):
    m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if mode == "frozen_backbone":
        for p in m.parameters():
            p.requires_grad = False          # freeze all conv layers; train only the head
    m.fc = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(m.fc.in_features, len(CLASSES)))
    return m.to(device)

def eval_f1_acc(model, dl):
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in dl:
            ps += model(x.to(device)).argmax(1).cpu().tolist(); ys += y.tolist()
    return f1_score(ys, ps, average="macro"), accuracy_score(ys, ps)

def train_one(mode):
    print(f"\n========== TRAIN_MODE = {mode} ==========")
    model = build_model(mode)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=WD)
    crit = nn.CrossEntropyLoss()
    ckpt_path = PROJECT / f"severity_best_{mode}.pt"
    best_f1, best_epoch, bad = -1, -1, 0
    for ep in range(1, EPOCHS+1):
        model.train(); tot = 0; ys, ps = [], []
        for x, y in train_dl:
            opt.zero_grad()
            out = model(x.to(device)); loss = crit(out, y.to(device))
            loss.backward(); opt.step(); tot += loss.item()*x.size(0)
            ps += out.argmax(1).detach().cpu().tolist(); ys += y.tolist()
        tr_f1, tr_acc = f1_score(ys, ps, average="macro"), accuracy_score(ys, ps)
        va_f1, va_acc = eval_f1_acc(model, val_dl)
        print(f"  ep {ep:02d}  loss={tot/len(train_ds):.3f} | "
              f"TRAIN acc={tr_acc:.3f} mF1={tr_f1:.3f} | VAL acc={va_acc:.3f} mF1={va_f1:.3f} | "
              f"gap(mF1)={tr_f1-va_f1:+.3f}")
        if va_f1 > best_f1:
            best_f1, best_epoch, bad = va_f1, ep, 0
            torch.save({"model": model.state_dict(), "classes": CLASSES, "mode": mode}, ckpt_path)
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"  early stop at {ep} (best val mF1={best_f1:.3f} @ ep {best_epoch})"); break
    print(f"  {mode}: best VAL macro-F1 = {best_f1:.3f} -> {ckpt_path}")
    return best_f1

results = {mode: train_one(mode) for mode in TRAIN_MODES}
winner = max(results, key=results.get)
print(f"\nVAL macro-F1 by mode: {results}")
print(f"WINNER (by val macro-F1): {winner}")


## Cell 4 - evaluate the winning mode vs the box-area heuristic baseline

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from torchvision import datasets
from torch.utils.data import DataLoader

print(f"VAL macro-F1 by mode: {results}")
print(f"Evaluating WINNER on val macro-F1: {winner}  (val mF1={results[winner]:.3f})")
model = build_model(winner)
ckpt = torch.load(PROJECT/f"severity_best_{winner}.pt", map_location=device)
model.load_state_dict(ckpt["model"]); model.eval()
CLASSES = ckpt["classes"]

def evaluate(folder, name, heuristic_acc):
    ds = datasets.ImageFolder(folder, eval_tf)
    dl = DataLoader(ds, 32, shuffle=False, num_workers=2)
    ys, ps = [], []
    with torch.no_grad():
        for x, y in dl:
            ps += model(x.to(device)).argmax(1).cpu().tolist(); ys += y.tolist()
    acc = accuracy_score(ys, ps)
    f1m = f1_score(ys, ps, average="macro")
    f1c = f1_score(ys, ps, average=None)
    cm  = confusion_matrix(ys, ps)
    print(f"\n=== {name} (n={len(ds)}) classes={CLASSES} ===")
    print(f"  CNN     : acc={acc:.3f}  macro-F1={f1m:.3f}  per-class F1={[round(float(v),3) for v in f1c]}")
    print(f"  heuristic: acc={heuristic_acc:.3f}  (box-area baseline)")
    print(f"  delta    : acc {acc-heuristic_acc:+.3f}")
    print("  confusion (rows=true, cols=pred):")
    for c, row in zip(CLASSES, cm): print(f"    {c:<7}{list(map(int,row))}")

evaluate(CROPS/"pothrgbd"/"test", "PothRGBD TEST (clean)", 0.41)   # heuristic macro-F1 0.42
evaluate(CROPS/"pothole440",      "Pothole440 (cross-dataset)", 0.28)  # heuristic macro-F1 0.16
print("\nNote: CNN input is RGB crop ONLY; crops resized 224x224 so box-area is not a shortcut.")
